# Lecture 3 — From Theory to Practice: Live Demo of Foundation Models with Hugging Face

**Presented by Dr. Fitsum Assamnew Andargie**  
**Duration:** 90 minutes  
**Environment:** Google Colab + Hugging Face

> **Central question:** What actually happens between writing an input and receiving an intelligent-looking output from a foundation model?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fassamnew/Foundation-Models/blob/main/Lecture_3_Foundation_Models_Hugging_Face_Colab.ipynb)

**Repository:** [github.com/fassamnew/Foundation-Models](https://github.com/fassamnew/Foundation-Models)

## Learning outcomes

By the end of this lab, participants should be able to:

1. distinguish **architecture, learned parameters, data, and hardware**;
2. load and use pretrained models from the **Hugging Face Hub**;
3. trace **text → tokens → representations → attention → prediction**;
4. distinguish **pretraining, inference, and fine-tuning**;
5. complete a short **gradient-based fine-tuning** experiment;
6. run a compact **generative language model**;
7. demonstrate **multimodal zero-shot classification** with CLIP; and
8. discuss **African-language coverage, compute access, licensing, bias, local data, and sovereignty**.

## 90-minute route

| Time | Activity |
|---|---|
| 0–7 min | Setup + hardware |
| 7–15 min | Pretrained model inference |
| 15–25 min | Self-supervised pretraining intuition |
| 25–38 min | Tokens, hidden states, attention |
| 38–55 min | Micro fine-tuning |
| 55–70 min | Generative model |
| 70–80 min | Multimodal CLIP |
| 80–87 min | African-language context |
| 87–90 min | Responsible deployment + wrap-up |

### Teaching pattern

For each demo use: **PREDICT → RUN → INSPECT → MODIFY → EXPLAIN**.

### How each demo is written

Each major section uses **Goal → Methodology → Expected outcome**, then **Discussion questions** and a **Self-exercise**. Before a new Hub model is downloaded, we summarize its **model card** and parameter scale, and call `show_model_card(...)`.


# 0. Setup

### Goal
Prepare a reproducible Colab environment with a GPU (when available), the required Python libraries, and a clear record of software + hardware versions.

### Methodology
1. Enable a GPU accelerator in Colab.
2. Install Hugging Face and supporting packages (with a safe Pillow pin).
3. Import libraries, fix a random seed, and print environment diagnostics.
4. Define small utilities for GPU memory checks and model-card inspection.

### Expected outcome
You should see `CUDA available: True` (or a clear CPU fallback message), package versions printed, and helper functions ready for the rest of the lab.

---

## Enable a GPU in Colab

A GPU makes model loading, fine-tuning, and generation much faster. Do this **before** running the install and setup cells below.

### Step 1 — Open the runtime settings

1. In the Colab menu bar at the top, click **Runtime**.
2. In the dropdown, click **Change runtime type**.

![Step 1: Runtime → Change runtime type](https://raw.githubusercontent.com/fassamnew/Foundation-Models/main/images/colab-gpu-step1-runtime-menu.png)

### Step 2 — Select a GPU and save

1. In the **Change runtime type** dialog, keep **Runtime type** as **Python 3**.
2. Open the **Hardware accelerator** dropdown and choose a GPU option (often **T4 GPU**; any available GPU is fine).
3. Click **Save**.

Colab may reconnect the session after you save. That is expected.

![Step 2: Hardware accelerator → GPU, then Save](https://raw.githubusercontent.com/fassamnew/Foundation-Models/main/images/colab-gpu-step2-change-runtime.png)

### Step 3 — Confirm the GPU is active

Check either of these:

- Top-right resource indicator: you should see **GPU** (not only RAM / Disk).
- Or open **Runtime → View resources** and confirm a GPU is listed.

Then run the setup cells below. You want output like:

```text
CUDA available: True
GPU: Tesla T4   # name may differ
```

If you see `CUDA available: False`, repeat Steps 1–2 and re-run the import cell.

![Step 3: Confirm GPU is connected](https://raw.githubusercontent.com/fassamnew/Foundation-Models/main/images/colab-gpu-step3-verify-gpu.png)

### If no GPU is available

Free Colab sometimes has no GPU capacity. You can still follow the notebook on CPU; the micro-training section already uses smaller settings when a GPU is missing. Training and generation will be slower.

## Install libraries

The notebook uses current Hugging Face interfaces. The install cell pins **Pillow ≥ 12.1.0** because Pillow 12.0.0 can break imports on Colab (`ImportError: cannot import name '_Ink'`).

After installing, run the next cell. If you still see a Pillow import error, choose **Runtime → Restart session**, then re-run from the import/version cell (skip the install unless packages are missing).

Before the workshop, run this notebook once in a fresh Colab runtime and record the package versions that worked. You can then pin those versions for maximum reproducibility.

### Discussion questions
1. Why does the same notebook behave differently on CPU vs GPU?
2. Which parts of an AI system are software, and which are hardware?

### Self-exercise
After the environment cell runs, write down: Python version, PyTorch version, Transformers version, GPU name (or `CPU`), and approximate GPU memory. Keep this note for reproducibility.


In [ ]:
# Pillow 12.0.0 breaks imports on Colab (_Ink missing). Use 12.1.0+.
%pip -q install -U transformers datasets accelerate huggingface_hub scikit-learn matplotlib "pillow>=12.1.0"

# If imports still fail after this cell, use Runtime → Restart session,
# then re-run from the next cell (you usually do not need to reinstall).


In [ ]:
import gc, platform, random, time
import numpy as np
import torch, transformers, datasets
from transformers import set_seed

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
else:
    print('CPU runtime detected; smaller training settings will be used.')

## Where is the intelligence?

### Goal
Separate **code**, **hardware**, **architecture**, and **learned parameters** so we do not treat “the model file” as the whole AI system.

### Methodology
Before loading any weights, name the components of an AI system and ask which ones change during fine-tuning vs inference.

### Expected outcome
A shared vocabulary for the rest of the lab: architecture ≠ trained model ≠ deployed system.

At this point we have code and hardware, but no model has been loaded.

**AI system = model architecture + learned parameters + runtime hardware + input data**

A Transformer architecture is not a trained foundation model, and a trained model still requires software and hardware to execute.

### Discussion questions
1. Which components change during fine-tuning?
2. Which components change during inference alone?
3. If two teams use the same architecture but different training data, is it the “same model”?

### Self-exercise
In one sentence each, define: architecture, parameters, tokenizer, and runtime hardware.


In [ ]:
from huggingface_hub import model_info

def cleanup():
    """Free unused Python objects and clear CUDA cache when possible."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def gpu_memory():
    """Print current CUDA memory allocation/reservation."""
    if not torch.cuda.is_available():
        print('GPU memory: not available')
        return
    print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB | Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

def show_model_card(model_id):
    """Inspect Hub metadata and reported parameter count BEFORE downloading weights."""
    info = model_info(model_id, files_metadata=True)
    card = info.card_data.to_dict() if info.card_data is not None else {}

    print('=' * 72)
    print('MODEL CARD SUMMARY (read this before downloading weights)')
    print('=' * 72)
    print('Model ID:     ', info.id)
    print('Hub URL:      ', f'https://huggingface.co/{info.id}')
    print('Pipeline:     ', info.pipeline_tag)
    print('Library:      ', getattr(info, 'library_name', None) or card.get('library_name'))
    print('License:      ', card.get('license'))
    print('Languages:    ', card.get('language'))
    print('Datasets:     ', card.get('datasets'))
    print('Base model:   ', card.get('base_model'))
    print('Tags:         ', (info.tags or [])[:18])

    # Parameter count from safetensors metadata when available on the Hub
    params = None
    st = getattr(info, 'safetensors', None)
    if st is not None:
        params = getattr(st, 'total', None)
        if params is None and isinstance(st, dict):
            params = st.get('total')
    if params is not None:
        print(f'Parameters:    {params:,}  ({params/1e6:.1f} M)')
        for bits in (32, 16, 8, 4):
            print(f'  ~weight storage @ {bits}-bit: {params * bits / 8 / 1e9:.3f} GB')
    else:
        print('Parameters:    not reported in Hub safetensors metadata for this repo')
        print('               (we will count tensors after loading when relevant)')

    if getattr(info, 'gated', False):
        print('Access:        GATED — you may need to accept terms on the Hub page')
    if 'custom_code' in (info.tags or []):
        print('Note:          Uses custom/remote code — review before trust_remote_code=True')
    print('=' * 72)
    return info

gpu_memory()
print('Helpers ready: cleanup(), gpu_memory(), show_model_card(model_id)')


# 1. USE — A pretrained model through `pipeline`

### Goal
Use a pretrained model for inference without training, and see the end-to-end path from text to label.

### Methodology
1. Read the model card and parameter count **before** downloading.
2. Load a Hugging Face `pipeline` for text classification.
3. Run a few English sentences and inspect labels + scores.

### Expected outcome
Each input receives a `POSITIVE` or `NEGATIVE` label with a confidence score. You should be able to explain that this was **inference**, not training.

---

## Meet the model before we download it

We start with **DistilBERT fine-tuned on SST-2** — a compact encoder trained first with self-supervised language modeling, then adapted to binary movie-review sentiment.

| Field | Value |
|---|---|
| Hub ID | [`distilbert/distilbert-base-uncased-finetuned-sst-2-english`](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) |
| Architecture | DistilBERT (encoder-only Transformer) |
| Task head | Sequence classification (2 labels) |
| Language | English |
| Approx. size | **~67 million parameters** (~66.96M reported on Hub) |
| Pretraining idea | Distilled from BERT; masked language modeling heritage |
| Fine-tuning data | SST-2 (GLUE) |
| Reported SST-2 accuracy | ~91.1% on validation (model card) |
| License | Apache-2.0 |

**Why this model for a workshop?** It is small enough to download quickly, well documented, and strong enough to show real pretrained behavior.

### Predict before running
Which examples should be positive, negative, or difficult to classify?

### Discussion questions
1. What does a score of `0.99` mean — and what does it *not* mean?
2. If the sentence is outside movie-review English, should we trust the label?

### Self-exercise
Write one clear positive sentence, one clear negative sentence, and one ambiguous sentence from your research domain. After running the next cells, compare your predictions with the model.


In [ ]:
SENTIMENT_MODEL = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
show_model_card(SENTIMENT_MODEL)


In [ ]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=SENTIMENT_MODEL,
    device=0 if torch.cuda.is_available() else -1,
)

examples = [
    'This workshop is extremely useful.',
    'The system failed and the experience was frustrating.',
    'The model worked, although the result was not as useful as I expected.'
]
for text in examples:
    print(text)
    print(classifier(text))
    print('-' * 70)


## What happened?

### Goal
Name each stage of inference so “the model answered” is not a black box.

### Methodology
Map the pipeline’s internal steps and contrast them with training.

### Expected outcome
You can state: we downloaded learned parameters and ran a forward pass only.

The pipeline performed:

**text → tokenization → tensors → Transformer → classification head → label**

We did **not** train the model. We downloaded learned parameters and performed **inference**.

### Discussion questions
1. Where could bias enter this pipeline even if your prompt looks fair?
2. Why might domain shift (agriculture / health / law) reduce reliability?

### Self-exercise
Change an input to a sentence from agriculture, health, education, engineering, or your own research domain. Compare a clear statement with an ambiguous one. Record both the label and the score.


# 2. PRETRAINING — Self-supervised learning intuition

### Goal
See how a foundation-model candidate learns from raw text via **masked language modeling**, without new human labels for every example.

### Methodology
1. Read the base DistilBERT model card.
2. Use a `fill-mask` pipeline to hide a token and rank plausible replacements.
3. Interpret top-k predictions as evidence of contextual pretraining, not task fine-tuning.

### Expected outcome
For each `[MASK]`, you get ranked token proposals with scores — illustrating a self-supervised learning signal.

---

## Meet the model before we download it

| Field | Value |
|---|---|
| Hub ID | [`distilbert/distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased) |
| Architecture | DistilBERT base, uncased |
| Objective (pretraining) | Knowledge distillation + MLM-style language modeling |
| Language | English |
| Approx. size | **~67 million parameters** |
| Vocabulary | WordPiece (~30k), lowercase |
| License | Apache-2.0 |
| Role in this lab | Base encoder for masking, tokenization, attention, and fine-tuning |

This is the **pretrained backbone**, not the SST-2 classifier. Same family, different head / adaptation stage.

### Discussion questions
1. Why can unlabeled text create a useful learning signal?
2. How is fill-mask different from the sentiment classifier we just used?

### Self-exercise
Before running, guess the top completion for each masked sentence below. Then compare your guess with the model.


In [ ]:
BASE_MODEL = 'distilbert/distilbert-base-uncased'
show_model_card(BASE_MODEL)


In [ ]:
fill_mask = pipeline(
    'fill-mask',
    model=BASE_MODEL,
    device=0 if torch.cuda.is_available() else -1,
)

for text in [
    'Artificial intelligence can help farmers [MASK] crop diseases.',
    'A foundation model can be adapted to many [MASK].'
]:
    print('\nINPUT:', text)
    for item in fill_mask(text, top_k=5):
        print(f"{item['token_str']!r:15s} score={item['score']:.4f}")


## Why this matters

### Goal
Connect self-supervised pretraining to the reusable “foundation” idea.

### Methodology
Use a simple diagram and critique model proposals for plausibility and bias.

### Expected outcome
You can explain: unlabeled corpus → pretrained model → many downstream adaptations.

```text
Large unlabeled corpus
        ↓
Self-supervised pretraining
        ↓
Reusable pretrained model
        ↓
 ┌────────────┬─────────────┬──────────────┐
 ↓            ↓             ↓
Sentiment   QA / NER     Other tasks
```

That reusability is central to the **foundation-model** idea.

### Discussion questions
1. Are high-scoring mask fills always factually correct?
2. What social or regional assumptions might appear in completions?

### Self-exercise
Replace a word in your own sentence with `[MASK]`. Ask whether the outputs are syntactically plausible, semantically plausible, and whether any assumptions or biases are visible.


# 3. OPEN — Tokens and language representation

### Goal
Show that Transformers never “see” raw characters first — they see **token IDs** — and that tokenization quality varies by language.

### Methodology
1. Tokenize an English sentence with DistilBERT’s WordPiece tokenizer.
2. Inspect tokens, IDs, and the attention mask.
3. Compare English vs Amharic token fragmentation on the same English-centric vocabulary.

### Expected outcome
You can read a tokenization printout and explain why unequal token counts matter for compute, context length, and representation quality.

We reuse **`distilbert/distilbert-base-uncased`** (already introduced). No new model download is required for this section — only the tokenizer.

### Discussion questions
1. Why might a morphologically rich or underserved language produce more tokens for the same meaning?
2. How could that affect latency, cost, and context-window usage?

### Self-exercise
After the English tokenization cell, predict whether Amharic will use fewer, similar, or many more tokens for a roughly parallel sentence.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
sentence = 'Foundation models learn reusable representations.'
enc = tokenizer(sentence, return_tensors='pt')

print('Text:', sentence)
print('Tokens:', tokenizer.tokenize(sentence))
print('Input IDs:', enc['input_ids'])
print('Attention mask:', enc['attention_mask'])
print('IDs as tokens:', tokenizer.convert_ids_to_tokens(enc['input_ids'][0]))

## Quick local-language check

### Goal
Make language coverage visible with a tiny, memorable comparison — not a full multilingual benchmark.

### Methodology
Tokenize parallel English and Amharic text with the same English DistilBERT tokenizer and compare token counts / pieces.

### Expected outcome
Amharic typically fragments more under an English WordPiece vocabulary, illustrating representation inequality at the tokenizer layer.

### Discussion questions
1. What happens downstream when a language is poorly represented in the tokenizer and pretraining corpus?
2. Would switching only the tokenizer (without changing pretraining data) fully solve the problem?

### Self-exercise
Add one more language you care about (Afaan Oromo, Tigrinya, French, etc.) to the dictionary in the next cell and compare token counts.


In [ ]:
texts = {
    'English': 'Artificial intelligence can support agriculture.',
    'Amharic': 'ሰው ሰራሽ አስተዋይነት ግብርናን መደገፍ ይችላል።'
}
for language, text in texts.items():
    pieces = tokenizer.tokenize(text)
    print(f'\n{language}')
    print('Text:', text)
    print('Tokens:', pieces)
    print('Number of token IDs:', len(tokenizer(text)['input_ids']))

## Reflection

Unequal tokenization can contribute to inefficient encoding, weaker representations, language mixing, missing cultural concepts, and uneven downstream performance.

### Self-exercise
Write 3 bullet points connecting tokenizer inequality to a deployment risk in education, health, or agriculture.


# 4. OPEN THE TRANSFORMER — Hidden states and attention

### Goal
Look inside the encoder: contextual hidden states and multi-head self-attention tensors.

### Methodology
1. Load DistilBERT with `AutoModel` (still the same ~67M-parameter base model).
2. Run a forward pass requesting `output_hidden_states` and `output_attentions`.
3. Interpret tensor shapes, then visualize averaged final-layer attention.

### Expected outcome
You can state the meaning of shapes like `batch × tokens × hidden` and `batch × heads × query × key`, and treat attention plots as **weight visualizations**, not full causal explanations.

### Model reminder
Same backbone: [`distilbert/distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased) (~67M parameters). Here we use the encoder only — no classification head.

### Discussion questions
1. Why should the representation of a word depend on its sentence context?
2. Why is an attention heatmap not the same as a complete explanation of the prediction?

### Self-exercise
Before running, sketch what shape you expect for the last hidden state of a short sentence.


In [ ]:
from transformers import AutoModel

encoder = AutoModel.from_pretrained(BASE_MODEL)
encoder.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder.to(device)

text = 'Foundation models learn reusable representations.'
inputs = tokenizer(text, return_tensors='pt')
inputs = {k:v.to(device) for k,v in inputs.items()}

with torch.no_grad():
    outputs = encoder(**inputs, output_hidden_states=True,
                      output_attentions=True, return_dict=True)

print('Last hidden-state shape:', tuple(outputs.last_hidden_state.shape))
print('Interpretation: batch × tokens × hidden dimension')
print('Transformer layers:', len(outputs.hidden_states)-1)
print('Attention tensors:', len(outputs.attentions))
print('One attention tensor:', tuple(outputs.attentions[-1].shape))
print('Interpretation: batch × heads × query tokens × key tokens')

## Visualize attention

### Goal
Make one attention pattern visible without overclaiming interpretability.

### Methodology
Average the final layer across heads and plot query→key weights for each token pair.

### Expected outcome
A heatmap where rows are query tokens and columns are key tokens. Use it as a discussion aid, not as proof of reasoning.

### Discussion questions
1. Which tokens attract the most attention in your plot, and is that linguistically sensible?
2. What could go wrong if a product team used only attention plots for safety claims?

### Self-exercise
Compare `The bank approved the loan.` with `We sat on the bank of the river.` Would you expect the contextual vector for `bank` to be identical? Re-run the encoder cells with each sentence and compare.


In [ ]:
import matplotlib.pyplot as plt

token_labels = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
att = outputs.attentions[-1][0].mean(dim=0).detach().cpu().numpy()

plt.figure(figsize=(9,7))
plt.imshow(att, aspect='auto')
plt.xticks(range(len(token_labels)), token_labels, rotation=75)
plt.yticks(range(len(token_labels)), token_labels)
plt.xlabel('Key token')
plt.ylabel('Query token')
plt.title('Final-layer self-attention averaged across heads')
plt.colorbar(label='Attention weight')
plt.tight_layout()
plt.show()

## Checkpoint

If the contextual representation of `bank` differed across sentences, that is the point of contextual encoders: meaning is computed from surrounding tokens, not from a single static embedding table lookup alone.


# 5. ADAPT — Micro fine-tuning in a few minutes

### Goal
Experience **genuine gradient-based adaptation** of a pretrained encoder to a labeled task — without pretending we trained a foundation model from scratch.

### Methodology
1. Load a small SST-2 subset from GLUE.
2. Attach a classification head to DistilBERT base (~67M parameters total).
3. Freeze embeddings + first four Transformer layers; train the top layers + head for 1 epoch.
4. Measure accuracy before vs after training and record wall-clock time.

### Expected outcome
Validation accuracy should move meaningfully above the untrained-head baseline, demonstrating transfer from pretraining.

---

## Starting checkpoint (already introduced)

| Field | Value |
|---|---|
| Hub ID | [`distilbert/distilbert-base-uncased`](https://huggingface.co/distilbert/distilbert-base-uncased) |
| Parameters | ~67M |
| New piece | Randomly initialized 2-class classification head |
| Data | SST-2 subset (`nyu-mll/glue`, config `sst2`) |
| Training style | Partial fine-tuning, 1 epoch, max length 96 |

> **Teaching question:** Why can a pretrained model adapt from hundreds of labeled examples when training from scratch would require vastly more data?

### Discussion questions
1. What is being reused from pretraining, and what is newly learned?
2. What accuracy would random guessing give on a balanced binary task?

### Self-exercise
Before training, write your guess for: (a) before-training accuracy, (b) after-training accuracy, (c) training time on your runtime.


In [ ]:
from datasets import load_dataset

TRAIN_N, VAL_N = (600, 200) if torch.cuda.is_available() else (160, 80)
raw_train = load_dataset('nyu-mll/glue', 'sst2', split='train')
raw_val = load_dataset('nyu-mll/glue', 'sst2', split='validation')
train_ds = raw_train.shuffle(seed=SEED).select(range(TRAIN_N))
val_ds = raw_val.shuffle(seed=SEED).select(range(VAL_N))

print('Training examples:', len(train_ds))
print('Validation examples:', len(val_ds))
print('Example:', train_ds[0])
print('Labels: 0=negative, 1=positive')

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

MAX_LENGTH = 96

def tokenize_batch(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=MAX_LENGTH)

tokenized_train = train_ds.map(tokenize_batch, batched=True)
tokenized_val = val_ds.map(tokenize_batch, batched=True)

train_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0:'NEGATIVE',1:'POSITIVE'},
    label2id={'NEGATIVE':0,'POSITIVE':1}
)

# Fast workshop mode: update only the top of the encoder + classification head.
train_model.distilbert.embeddings.requires_grad_(False)
for layer in train_model.distilbert.transformer.layer[:4]:
    layer.requires_grad_(False)

total = sum(p.numel() for p in train_model.parameters())
trainable = sum(p.numel() for p in train_model.parameters() if p.requires_grad)
print(f'Total parameters: {total:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'Trainable fraction: {100*trainable/total:.2f}%')

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

## What is being trained?

### Goal
Identify which tensors receive gradients in this fast configuration.

### Methodology
Inspect total vs trainable parameter counts printed by the previous cell, and relate them to the frozen/unfrozen layers.

### Expected outcome
You can point to: frozen lower encoder, trainable upper encoder + classification head.

```text
Pretrained DistilBERT encoder
        ↓
Task-specific classification head
        ↓
NEGATIVE / POSITIVE
```

The task-specific classification head starts newly initialized. Most of the linguistic knowledge comes from pretraining.

### Discussion questions
1. Why freeze early layers in a workshop setting?
2. What trade-offs appear if you unfreeze everything?

### Self-exercise
From the printed counts, compute the trainable fraction and note it beside your accuracy guesses.


In [ ]:
from transformers import Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': float((preds == labels).mean())}

args = TrainingArguments(
    output_dir='./workshop_distilbert_sst2',
    learning_rate=5e-5,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 8,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 16,
    num_train_epochs=1,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=10,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=train_model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
print('Trainer ready.')

## Baseline before training

### Goal
Measure the untrained head so improvement is attributable to fine-tuning.

### Methodology
Call `trainer.evaluate()` once before `trainer.train()`.

### Expected outcome
Accuracy near chance (~0.5 on balanced binary data) or otherwise clearly weaker than the post-training score.

### Discussion questions
1. If before-training accuracy is already high, what might that imply?
2. Why evaluate on a held-out validation split rather than training examples alone?


In [ ]:
before = trainer.evaluate()
print(f"Before training accuracy: {before['eval_accuracy']:.3f}")
print(f"Before training loss: {before['eval_loss']:.3f}")

## Train and measure the wall-clock time

### Goal
Connect adaptation to **compute cost**, not only accuracy.

### Methodology
Time one epoch of `trainer.train()`, then re-evaluate.

### Expected outcome
A wall-clock duration plus an after-training accuracy/loss. Exact numbers vary by hardware.

### Self-exercise
Record: hardware (GPU/CPU), training seconds, before accuracy, after accuracy.


In [ ]:
start = time.perf_counter()
train_result = trainer.train()
elapsed = time.perf_counter() - start
print(f'Training time: {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)')

In [ ]:
after = trainer.evaluate()
print('Before training accuracy:', round(before['eval_accuracy'],3))
print('After training accuracy: ', round(after['eval_accuracy'],3))
print('After training loss:     ', round(after['eval_loss'],3))

## Interpret the result

### Goal
State the pretraining → fine-tuning relationship in one diagram and one sentence.

### Methodology
Compare before/after metrics and discuss what would change if we scaled data, epochs, or trainable layers.

### Expected outcome
You can explain: we reused general representations; we did not invent language knowledge in this session.

```text
PRETRAINING                     FINE-TUNING
Broad text data                 Small labeled task data
      ↓                               ↓
General representations  →  Task-adapted representations
```

### Discussion questions
1. Is higher validation accuracy sufficient for deployment in a new domain?
2. How would you detect overfitting on a tiny subset?

### Self-exercise
If time permits, increase the dataset, unfreeze more layers, or add another epoch and compare **accuracy, training time, and memory**.


# 6. GENERATE — A compact instruction-tuned language model

### Goal
Run open-ended **next-token generation** with a small instruction-tuned causal LM, then connect parameter count to memory.

### Methodology
1. Free earlier encoder/trainer objects to reclaim GPU memory.
2. Read the SmolLM2 model card and parameter estimate **before** downloading.
3. Generate with greedy decoding, then with sampling.
4. Count parameters and estimate weight storage at 32/16/8/4-bit.

### Expected outcome
A short assistant-style answer, a second more varied sample, and a clear parameters→memory table.

---

## Meet the model before we download it

| Field | Value |
|---|---|
| Hub ID | [`HuggingFaceTB/SmolLM2-360M-Instruct`](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) |
| Family | SmolLM2 (135M / 360M / 1.7B) |
| Approx. size | **~360 million parameters** (~0.4B class) |
| Type | Decoder-only causal LM, **instruction-tuned** (SFT + preference optimization lineage) |
| Why here | Small enough for Colab, large enough to show chat-style generation |
| Paper | https://arxiv.org/abs/2502.02737 |

A causal LM predicts the next token repeatedly. Instruction tuning makes chat-format prompts more useful, but it does **not** guarantee factual correctness.

### Discussion questions
1. How is generation different from DistilBERT classification?
2. If the answer sounds fluent, does that mean it is grounded in verified agricultural knowledge?

### Self-exercise
Rewrite the user prompt for a domain you know well. Predict whether greedy vs sampled decoding will differ more in style or in facts.


In [ ]:
del trainer, train_model, encoder
cleanup(); gpu_memory()

In [ ]:
GEN_MODEL = 'HuggingFaceTB/SmolLM2-360M-Instruct'
show_model_card(GEN_MODEL)


In [ ]:
generator = pipeline(
    'text-generation',
    model=GEN_MODEL,
    device_map='auto',
    dtype='auto',
)

# Count parameters on the loaded model (authoritative for this session)
n_params = sum(p.numel() for p in generator.model.parameters())
print(f'Loaded parameters: {n_params:,} ({n_params/1e6:.1f} M)')

messages = [{
    'role': 'user',
    'content': 'Give three practical ways artificial intelligence could support a smallholder farmer. Keep the answer concise.'
}]

out = generator(messages, max_new_tokens=120, do_sample=False)
print(out[0]['generated_text'][-1]['content'])


## Generation settings are not training

### Goal
Separate **decoding choices** from **learned weights**.

### Methodology
Re-run the same prompt with sampling (`temperature`, `top_p`) and compare outputs.

### Expected outcome
A second continuation that may differ in wording. Weights are unchanged.

At inference time the model repeatedly produces a probability distribution over next tokens. `do_sample=False` is more deterministic; sampling allows alternative continuations.

### Discussion questions
1. Which settings would you prefer for a grading assistant vs a brainstorming assistant?
2. Can decoding settings fix a model that lacks the right knowledge?

### Self-exercise
Keep the prompt fixed. Try `temperature=0.2` and `temperature=1.0`. Note differences in specificity and stability.


In [ ]:
out2 = generator(messages, max_new_tokens=120,
                 do_sample=True, temperature=0.9, top_p=0.9)
print(out2[0]['generated_text'][-1]['content'])

## Connect parameters to hardware

### Goal
Translate “360M parameters” into approximate **weight storage**, and remember that training/inference need more than weights alone.

### Methodology
Use the loaded parameter count and estimate bytes at common precisions.

### Expected outcome
A table of GB estimates for 32/16/8/4-bit weights, plus awareness of activations, KV cache, and optimizer states.

### Discussion questions
1. Why can inference memory exceed the weight-only estimate?
2. Which systems techniques reduce memory without changing task code much (mixed precision, quantization, etc.)?

### Self-exercise
Using your GPU’s memory printout from earlier, ask whether FP16 weights of this model should fit. Then explain what else still consumes memory.


In [ ]:
params = sum(p.numel() for p in generator.model.parameters())
print(f'Parameters: {params/1e6:.1f} million')
for bits in [32,16,8,4]:
    print(f'{bits:>2}-bit weights ≈ {params*bits/8/1e9:.3f} GB')

## Systems takeaway

Modern foundation models are **systems-engineering achievements**, not architecture diagrams alone. Mixed precision, quantization, memory-efficient attention, distributed training, and accelerator design all shape what is deployable.

### Self-exercise
Name one technique that primarily saves **memory**, one that primarily saves **wall-clock time**, and one that primarily improves **model quality** — they are not always the same lever.


# 7. MULTIMODAL — CLIP zero-shot image classification

### Goal
Classify an image by comparing it to **text descriptions**, without training a new fixed N-class head.

### Methodology
1. Free the generative model to reclaim memory.
2. Load a sample image.
3. Read the CLIP model card and parameter footprint.
4. Run zero-shot image classification with candidate label strings.

### Expected outcome
Ranked labels with scores for the image. The key idea: shared image–text representation space.

> **Timing note:** this requires another model download. If workshop bandwidth is weak, make this instructor-led or assign it as a post-session extension.

---

## Meet the model before we download it

| Field | Value |
|---|---|
| Hub ID | [`openai/clip-vit-base-patch32`](https://huggingface.co/openai/clip-vit-base-patch32) |
| Architecture | CLIP: Vision Transformer (ViT-B/32) + text Transformer |
| Task | Zero-shot image classification / image–text matching |
| Approx. size | **~150 million parameters** (ViT-B/32 CLIP; exact count printed after load) |
| Training idea | Contrastive learning on image–text pairs |
| Paper | Radford et al., 2021 — https://arxiv.org/abs/2103.00020 |

CLIP does not require you to train a new classifier for every label set: you supply label text at inference time.

### Discussion questions
1. How can changing only the label wording change the prediction?
2. What fails if labels or images are culturally or geographically mismatched?

### Self-exercise
Before running, rank the candidate labels yourself for the sample image. Compare with CLIP’s ranking.


In [ ]:
del generator
cleanup(); gpu_memory()

In [ ]:
from PIL import Image
import requests
from io import BytesIO
from IPython.display import display

url = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png'
r = requests.get(url, timeout=30); r.raise_for_status()
image = Image.open(BytesIO(r.content)).convert('RGB')
display(image)

In [ ]:
CLIP_MODEL = 'openai/clip-vit-base-patch32'
show_model_card(CLIP_MODEL)


In [ ]:
clip_classifier = pipeline(
    'zero-shot-image-classification',
    model=CLIP_MODEL,
    device=0 if torch.cuda.is_available() else -1,
)

n_params = sum(p.numel() for p in clip_classifier.model.parameters())
print(f'Loaded CLIP parameters: {n_params:,} ({n_params/1e6:.1f} M)')

labels = [
    'a photograph of animals',
    'a photograph of people',
    'a photograph of a city',
    'a photograph of agricultural land',
    'a photograph taken indoors'
]
results = clip_classifier(image, candidate_labels=labels)
for item in results:
    print(f"{item['label']:<40s} {item['score']:.4f}")


## What is the key idea?

### Goal
Name the multimodal shift: from fixed task heads to **reusable image–text representations**.

### Methodology
Reflect on the ranked scores, then optionally try your own image with new label strings.

### Expected outcome
You can explain zero-shot classification as similarity in a joint embedding space — still contingent on training data coverage.

We did not train a five-class image classifier. The model compares learned image and text representations.

### Discussion questions
1. Is zero-shot the same as “no data was ever used”?
2. When would you still fine-tune a vision model instead of using CLIP-style prompting?

### Self-exercise
```python
from google.colab import files
uploaded = files.upload()
path = next(iter(uploaded))
my_image = Image.open(path).convert('RGB')
display(my_image)
clip_classifier(my_image, candidate_labels=['...', '...'])
```

Use only images you have permission to use. Design labels that could fail (synonyms, negation, rare local objects) and observe the scores.


# 8. AFRICAN CONTEXT — Inspect before you deploy

### Goal
Practice **model-card-first** evaluation for an African-language LLM, including languages, license, and trust boundaries — without immediately downloading large weights.

### Methodology
1. Call `show_model_card` / Hub metadata for InkubaLM-0.4B.
2. Compare advertised languages with Ethiopian deployment needs.
3. Discuss gated access, custom code, and local evaluation requirements.

### Expected outcome
A written judgment: promising research artifact ≠ automatically deployable local assistant.

---

## Meet the model before any download

| Field | Value |
|---|---|
| Hub ID | [`lelapa/InkubaLM-0.4B`](https://huggingface.co/lelapa/InkubaLM-0.4B) |
| Approx. size | **~0.4B parameters** |
| Type | Causal LM (LLaMA-style lineage; custom code on Hub) |
| Languages (card) | English, Swahili, Zulu, Xhosa, Hausa, Yoruba |
| Dataset tag | `lelapa/Inkuba-Mono` |
| License | **CC-BY-NC-4.0** (non-commercial — read carefully) |
| Access | Often **gated** — accept terms on the Hub if prompted |
| Paper | Tonja et al., 2024 — https://arxiv.org/abs/2408.17024 |

**Important:** InkubaLM is a strong teaching example for African NLP progress, but the card languages above do **not** automatically include Amharic, Afaan Oromo, or Tigrinya. Always verify coverage for your deployment languages.

### Discussion questions
1. What is the difference between “open weights” and “appropriate for my institution”?
2. Why inspect the card before `from_pretrained`?

### Self-exercise
Fill a one-row audit: languages needed, languages claimed, license OK for use?, evaluation evidence?, compute fit?


In [ ]:
AFRICAN_MODEL = 'lelapa/InkubaLM-0.4B'
info = show_model_card(AFRICAN_MODEL)

# Extra detail from the card object when present
card = info.card_data.to_dict() if info.card_data is not None else {}
print('\nDeployment-oriented checklist fields')
print('License:  ', card.get('license'))
print('Languages:', card.get('language'))
print('Datasets: ', card.get('datasets'))
print('Gated:    ', getattr(info, 'gated', None))
print('Custom code tag:', 'custom_code' in (info.tags or []))


## Questions for an Ethiopian deployment

### Goal
Turn model-card reading into a deployment gate review.

### Methodology
Answer each question with evidence from the card, paper, or “unknown — needs local eval”.

### Expected outcome
A reasoned go / no-go / research-only recommendation.

1. Is **Amharic** represented? What about Afaan Oromo, Tigrinya, Somali, Sidama, Afar, or other required languages?
2. What pretraining data shaped the model?
3. What license applies for your institution’s intended use (note NC terms when present)?
4. What evidence exists for the **exact intended task**?
5. What local evaluation is still necessary?
6. Can the model run economically on available infrastructure (~0.4B is smaller than frontier LMs, but still not free)?
7. Who controls the data, model hosting, updates, and logs?

> **Open model ≠ locally appropriate model.**

### Optional instructor extension

InkubaLM’s repository currently uses remote custom model code. Only use `trust_remote_code=True` after reviewing and trusting that repository.

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
model_id = 'lelapa/InkubaLM-0.4B'
inkuba_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
inkuba_model = AutoModelForCausalLM.from_pretrained(
    model_id, trust_remote_code=True, device_map='auto', dtype='auto'
)
n = sum(p.numel() for p in inkuba_model.parameters())
print(f'Loaded parameters: {n:,} ({n/1e6:.1f} M)')
```

### Self-exercise
If you load the model, prompt it in a language from its card and in Amharic. Document qualitative differences without treating either run as a formal benchmark.


# 9. RESPONSIBLE DEPLOYMENT CHALLENGE

### Goal
Apply the lab’s technical concepts to a realistic deployment decision.

### Methodology
Imagine an institution wants an AI assistant that gives crop-disease guidance to farmers in multiple Ethiopian languages. A promising model exists on Hugging Face. Score it on the dimensions below.

### Expected outcome
A structured recommendation: deploy, pilot with oversight, or do not deploy — with missing evidence listed.

| Dimension | Question |
|---|---|
| Task fit | Was it evaluated for this task? |
| Language | Does it represent the intended languages and varieties? |
| Data | What relevant local data is absent? |
| Reliability | How will hallucinations and uncertainty be evaluated? |
| Human oversight | Which outputs require expert review? |
| Safety | What happens when advice is wrong? |
| Bias | Which populations or regions may be poorly represented? |
| Privacy | What user or institutional data enters the system? |
| Sovereignty | Where are data and inference processed? |
| Compute | Can it run reliably and affordably? |
| License | Is the intended use permitted? |
| Monitoring | How will failures be detected after deployment? |

## Final systems view

```text
                 FOUNDATION MODEL
                       │
       ┌───────────────┼────────────────┐
       │               │                │
     DATA          ARCHITECTURE       COMPUTE
       └───────────────┼────────────────┘
                       ↓
                  PRETRAINING
                       ↓
             Reusable representations
                       ↓
       domain adaptation + local data
                       +
                   evaluation
                       +
                human oversight
                       +
              responsible governance
                       ↓
                DEPLOYED AI SYSTEM
```

> **Main takeaway:** A foundation model is not the finished AI system. It is a computational foundation from which a system can be built.

### Self-exercise
Pick one Hugging Face model relevant to your work. Complete the table above in 15 minutes using only the model card and README.


# 10. What we did

### Goal
Consolidate the lab into three verbs and the models we actually touched.

### Methodology
Retell the path from inference → pretraining intuition → internals → adaptation → generation → multimodality → local audit.

### Expected outcome
You can explain each stage and cite approximate parameter scales.

### USE
`pretrained model → input → output`  
Models: DistilBERT-SST2 (~67M), DistilBERT base (~67M)

### OPEN
`text → tokenizer → token IDs → representations → attention → Transformer → prediction`

### ADAPT
`pretrained model + small labeled dataset → gradient-based fine-tuning → downstream model`

### GENERATE / MULTIMODAL / AUDIT
SmolLM2-360M-Instruct (~360M), CLIP ViT-B/32 (~150M), InkubaLM-0.4B card (~0.4B)

### Discussion questions
1. Which stage changes parameters? Which stages only run a forward pass?
2. Where did language coverage become a first-class concern?


# 11. Post-workshop challenges

### Goal
Extend the lab with deliberate experiments you can finish after class.

### Methodology
Pick one challenge. Write goal / method / outcome before coding. Include a model-card screenshot or printed `show_model_card` output in your notes.

1. **Compare tokenizers:** test English, Amharic, Afaan Oromo, or another language across English-only and multilingual models. Report token counts and a qualitative error analysis.
2. **Full vs partial fine-tuning:** unfreeze all DistilBERT layers and compare time, memory, and validation accuracy.
3. **Parameter-efficient fine-tuning:** explore LoRA/PEFT and compare trainable parameter counts vs full fine-tuning.
4. **Domain evaluation:** build a small, carefully reviewed evaluation set from your domain without exposing confidential data.
5. **Model-card audit:** for a Hub model you might deploy, extract parameters, license, languages, datasets, and risks; list local validation still required.

### Self-exercise (minimum)
Complete challenge 5 for any one model used today and one additional model of your choice.


# 12. References and resources

## Selected academic references

- Rumelhart, Hinton & Williams (1986) — Backpropagation.
- Krizhevsky, Sutskever & Hinton (2012) — AlexNet / ImageNet.
- Mikolov et al. (2013) — word2vec.
- Bahdanau, Cho & Bengio (2014) — Attention for NMT.
- Jouppi et al. (2017) — TPU.
- Micikevicius et al. (2017) — Mixed precision training.
- Vaswani et al. (2017) — *Attention Is All You Need*.
- Devlin et al. (2018) — BERT.
- Radford et al. (2018) — GPT.
- Shoeybi et al. (2019) — Megatron-LM.
- Brown et al. (2020) — GPT-3.
- Dosovitskiy et al. (2020) — Vision Transformer.
- Kaplan et al. (2020) — Scaling laws.
- Rajbhandari et al. (2020) — ZeRO.
- Wolf et al. (2020) — Transformers library.
- Bommasani et al. (2021) — Foundation models.
- Jumper et al. (2021) — AlphaFold2.
- Radford et al. (2021) — CLIP.
- Avsec et al. (2021) — Enformer.
- Hoffmann et al. (2022) — Chinchilla.
- Dao et al. (2022) — FlashAttention.
- Lam et al. (2023) — GraphCast.
- Merchant et al. (2023) — GNoME.
- Singhal et al. (2023) — Med-PaLM.
- Jakubik et al. (2023) — Prithvi geospatial FM.
- Lin et al. (2023) — ESM-2.
- Tonja et al. (2024) — InkubaLM.

## Hugging Face resources used

- https://huggingface.co/docs/transformers/
- https://huggingface.co/docs/datasets/
- https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
- https://huggingface.co/distilbert/distilbert-base-uncased
- https://huggingface.co/datasets/nyu-mll/glue
- https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct
- https://huggingface.co/openai/clip-vit-base-patch32
- https://huggingface.co/lelapa/InkubaLM-0.4B

## GitHub hosting

This notebook is hosted at:

- **Repo:** https://github.com/fassamnew/Foundation-Models
- **Open in Colab:** https://colab.research.google.com/github/fassamnew/Foundation-Models/blob/main/Lecture_3_Foundation_Models_Hugging_Face_Colab.ipynb

Repository structure:

```text
Foundation-Models/
├── images/
│   ├── colab-gpu-step1-runtime-menu.png
│   ├── colab-gpu-step2-change-runtime.png
│   └── colab-gpu-step3-verify-gpu.png
└── Lecture_3_Foundation_Models_Hugging_Face_Colab.ipynb
```

Before delivery:

1. run the notebook from a fresh Colab GPU runtime;
2. note the package versions printed at startup;
3. optionally pin those exact versions in the installation cell; and
4. retain an already-executed instructor copy as a fallback for poor connectivity.

## Model cards to re-read

Always open the Hub page before downloading:

- DistilBERT SST-2: https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english
- DistilBERT base: https://huggingface.co/distilbert/distilbert-base-uncased
- SmolLM2-360M-Instruct: https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct
- CLIP ViT-B/32: https://huggingface.co/openai/clip-vit-base-patch32
- InkubaLM-0.4B: https://huggingface.co/lelapa/InkubaLM-0.4B
